# 01 - Download BTCUSDT 15m data from Binance

Acquires the full public BTCUSDT 15-minute history and writes
`data/raw/btcusdt_15m_raw.csv`.

**Method (most efficient first)**

| Priority | Source | Why |
|---|---|---|
| 1 | `data.binance.vision` monthly ZIP | one request per month instead of ~30 REST calls |
| 2 | `data.binance.vision` daily ZIP | covers the current, still-open month |
| 3 | REST `/api/v3/klines` | only the newest candles not yet archived |

No API key is needed - this is public market data.

**Handled automatically:** pagination, rate limits, retries with backoff, the
Binance millisecond -> microsecond timestamp switch (2025-01), duplicate
candles across overlapping sources, chronological ordering, malformed rows and
invalid OHLC values.

Gaps are **never** filled here - see notebook 02.

In [ ]:
import sys, os, json
from pathlib import Path

# Make src/ importable whether this notebook runs from notebooks/ or the root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

In [ ]:
from data_download import download_all

# First run downloads ~110 monthly archives (~1-2 minutes).
# Re-running reuses the cache and refreshes only the most recent months.
raw, report = download_all()
print(report.render())

In [ ]:
import pandas as pd

raw = pd.read_csv(ROOT / "data" / "raw" / "btcusdt_15m_raw.csv")
print("rows    :", f"{len(raw):,}")
print("columns :", list(raw.columns))
print("from    :", raw["timestamp"].iloc[0])
print("to      :", raw["timestamp"].iloc[-1])
raw.head()

In [ ]:
# Confirm the 15-minute grid and strictly increasing timestamps.
ts = pd.to_datetime(raw["timestamp"], utc=True)
print(ts.diff().dropna().value_counts().head(10))
print("\nmonotonic :", ts.is_monotonic_increasing)
print("duplicates:", int(ts.duplicated().sum()))

### Expected result

Roughly 310k-320k candles spanning 2017-08 to the present. Any missing candles
are reported by notebook 02 and handled by segment-aware windowing - never by
interpolation.